# Decomposing Bilinear MLP Weights into Human Concepts

Experiments for the MARS V (Goodfire stream) applicant task. See `METHOD_REFERENCE.md` for the theory and experimental design.

In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import pandas as pd
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from tqdm import tqdm
from torch.optim.lr_scheduler import CosineAnnealingLR

from image import Model, MNIST
from image.sparse import Model as Sparse
from kornia.augmentation import RandomGaussianNoise

pio.renderers.default = "notebook"
device = "cpu:0"

In [ ]:
RANK = 64
N_STEPS = 200
LR = 0.02
SEED = 42
K_VIS = 8

ALPHAS_L1 = [0.001, 0.01, 0.1]

torch.manual_seed(SEED);

## MNIST classifier

Train the bilinear MNIST model with Gaussian noise augmentation. The interaction tensor `B` of this model is the target of every decomposition below.

In [ ]:
import os

MNIST_CACHE = "data/mnist_bilinear.pt"

train, test = MNIST(train=True, device=device), MNIST(train=False, device=device)
model = Model.from_config(epochs=20).to(device)
model.fit(train, test, RandomGaussianNoise(std=0.4))


##Helpers

`fit_decomposition` is parameterised over the three interventions (L1, symmetric tie, non-negativity) so each experiment is a one-line call. `evaluate` returns cosine similarity to `B` plus classification accuracy. `visualize_decomposition` renders `L+R`, `L-R` and the per-class `D` bars for the top-`k` components.

In [ ]:
def fit_decomposition(model, rank=RANK, n_steps=N_STEPS, lr=LR,
                      alpha_l1=0.0, alpha_l1_d=0.0, symmetric=False,
                      nonneg=False, seed=SEED, verbose=True):
    """Fit a Sparse CP-style decomposition with optional sparsity / symmetry / non-negativity priors.

    Upstream `Sparse` exposes params as `.left`, `.right`, `.down` — not `.l`, `.r`, `.d`.
    When `nonneg=True` the squaring is applied to the *reconstruction* (per METHOD_REFERENCE §10),
    then baked back into `sparse.left` / `sparse.right` at the end so downstream evaluate / visualize
    see the positive factors that were actually trained."""
    torch.manual_seed(seed)
    sparse = Sparse.from_config(rank=rank).to(device)

    optimizer = torch.optim.Muon(sparse.parameters(), lr=lr, momentum=0.95)
    scheduler = CosineAnnealingLR(optimizer, T_max=n_steps)

    B_target = reconstruct_B_target(model).detach() if nonneg else None
    target_norm = B_target.norm() if nonneg else None

    torch.set_grad_enabled(True)
    pbar = tqdm(range(n_steps)) if verbose else range(n_steps)

    for _ in pbar:
        if symmetric:
            with torch.no_grad():
                sparse.right.data = sparse.left.data.clone()

        if nonneg:
            L_eff = sparse.left ** 2
            R_eff = sparse.right ** 2
            t = torch.einsum("cr,ir,jr->cij", sparse.down, L_eff, R_eff)
            B_hat = 0.5 * (t + t.transpose(-1, -2))
            sim = (B_target * B_hat).sum() / (target_norm * B_hat.norm())
        else:
            sim = sparse.similarity(model)
            L_eff, R_eff = sparse.left, sparse.right

        loss = 1 - sim

        if alpha_l1 > 0:
            loss = loss + alpha_l1 * (L_eff.abs().mean() + R_eff.abs().mean())

        if alpha_l1_d > 0:
            loss = loss + alpha_l1_d * sparse.down.abs().mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        scheduler.step()

        if verbose and hasattr(pbar, "set_description"):
            pbar.set_description(f"loss: {loss.item():.4f} sim: {sim.item():.4f}")

    if nonneg:
        with torch.no_grad():
            sparse.left.data = sparse.left.data ** 2
            sparse.right.data = sparse.right.data ** 2

    torch.set_grad_enabled(False)
    return sparse


def evaluate(sparse_model, model, test):
    with torch.no_grad():
        orig_acc = (model(test.x).argmax(-1) == test.y).float().mean().item()
        sparse_acc = (sparse_model(test.x).argmax(-1) == test.y).float().mean().item()
        sim = sparse_model.similarity(model).item()
    return {"orig_acc": orig_acc, "sparse_acc": sparse_acc, "similarity": sim}


def visualize_decomposition(sparse_model, title="", save_path=None, k=K_VIS):
    plus, minus, down, _sigma = sparse_model.decompose()
    plus, minus, down = plus.cpu(), minus.cpu(), down.cpu()

    fig = make_subplots(rows=3, cols=k, row_titles=["L+R", "L-R", "D"], vertical_spacing=0.08)
    for i in range(k):
        params = dict(showscale=False, colorscale="RdBu", zmid=0)
        fig.add_heatmap(z=plus[:, i].view(28, 28).flip(0), **params, row=1, col=i + 1)
        fig.add_heatmap(z=minus[:, i].view(28, 28).flip(0), **params, row=2, col=i + 1)
        fig.add_bar(y=down[:, i], marker_color=["gray"] * 10, showlegend=False, row=3, col=i + 1)

    fig.update_xaxes(visible=False).update_yaxes(visible=False)
    fig.update_layout(width=k * 110, height=400, margin=dict(l=40, r=0, b=0, t=40),
                      template="plotly_white", title=title)

    if save_path:
        try:
            fig.write_image(save_path)
        except (ValueError, ImportError) as e:
            print(f"skipped saving {save_path}: {e}")
    return fig


def reconstruct_B_target(orig):
    """Build the symmetrized interaction tensor B[c,i,j] from a trained Model.

    This is exactly what `Sparse.similarity()` reconstructs internally — we expose it
    so Frobenius-loss experiments (Exp 5) can train against it directly."""
    wl, wr = orig.w_lr[0].unbind()
    wl, wr = wl @ orig.w_e, wr @ orig.w_e
    t = torch.einsum("co,oi,oj->cij", orig.w_u, wl, wr)
    return 0.5 * (t + t.transpose(-1, -2))


results = []


In [ ]:
torch.manual_seed(123)
test_sparse = Sparse.from_config(rank=RANK).to(device)

with torch.no_grad():
    B_target = reconstruct_B_target(model)
    B_hat = test_sparse.tensor()

    # 1. Manual cosine matches the API
    manual_cos = ((B_target * B_hat).sum() / (B_target.norm() * B_hat.norm())).item()
    api_cos = test_sparse.similarity(model).item()
    assert abs(manual_cos - api_cos) < 1e-5, f"similarity API mismatch: manual={manual_cos:.6f} api={api_cos:.6f}"

    # 2. B_target is symmetric in (i, j)
    sym_err = (B_target - B_target.transpose(-1, -2)).abs().max().item()
    assert sym_err < 1e-5, f"B_target not symmetric: max asymmetry = {sym_err}"

    # 3. Bounded in [-1, 1]
    assert -1.0 - 1e-5 <= api_cos <= 1.0 + 1e-5, f"cosine out of range: {api_cos}"

    # 4. Scale invariance — the property responsible for the L1 degeneracy in Exp 2
    scaled = 7.3 * B_hat
    cos_scaled = ((B_target * scaled).sum() / (B_target.norm() * scaled.norm())).item()
    assert abs(cos_scaled - manual_cos) < 1e-5, "cosine should be scale-invariant"

print(f"cosine-sim sanity test passed (random sparse → cos = {api_cos:.4f})")
print(f"B_target shape: {tuple(B_target.shape)}, ||B_target||_F = {B_target.norm().item():.2f}")
print(f"B_target symmetry error: {sym_err:.2e}")


## Experiment 1 — Baseline

Pure CP decomposition, cosine loss, no priors. Establishes what *sharing alone* gets you: components are free to overlap across classes via the shared `D` factor, but nothing else encourages interpretability.

In [ ]:
sparse_base = fit_decomposition(model, alpha_l1=0.0)
metrics_base = evaluate(sparse_base, model, test)
fig_base = visualize_decomposition(sparse_base, title="Baseline (no priors)",
                                   save_path="fig_baseline.png")
print(metrics_base)
results.append({"name": "baseline", "sparse": sparse_base, "metrics": metrics_base})
fig_base.show()

## Experiment 5 — Honest dictionary learning

Overcomplete CP (`R = 128` vs embed `d = 64`), unit-norm atom columns, Frobenius reconstruction loss, L1 on both atoms (`L,R`) and code (`D`). This is the classical Olshausen-Field / sparse-coding setup applied to the bilinear interaction tensor — the framing the L1 sweep in Exp 2 only gestures at. See `experiment_plans/experiment_05_dictionary_learning.md`.

In [ ]:
OC_RANK = 128
N_STEPS_DL = 400
ALPHA_ATOM = 0.001  # was 0.01 — too aggressive
SEED = 0

def fit_dictionary_learning(model, oc_rank=OC_RANK, n_steps=N_STEPS_DL,
                            alpha_atom=ALPHA_ATOM,
                            lr=1e-2, seed=SEED, verbose=True):

    torch.manual_seed(seed)
    sparse = Sparse.from_config(rank=oc_rank).to(device)
    B_target = reconstruct_B_target(model).detach()
    
    # Diagnostic: check signal scale
    print(f"B_target stats: mean={B_target.abs().mean().item():.6f}, "
          f"max={B_target.abs().max().item():.6f}, "
          f"std={B_target.std().item():.6f}")
    
    # Initialise atoms to unit norm
    with torch.no_grad():
        sparse.left.data /= sparse.left.data.norm(dim=0, keepdim=True).clamp_min(1e-8)
        sparse.right.data /= sparse.right.data.norm(dim=0, keepdim=True).clamp_min(1e-8)

    opt = torch.optim.Adam(sparse.parameters(), lr=lr)
    torch.set_grad_enabled(True)
    pbar = tqdm(range(n_steps)) if verbose else range(n_steps)

    for _ in pbar:
        B_hat = sparse.tensor()  # already symmetrized
        recon = (B_target - B_hat).pow(2).mean()
        reg_atom = alpha_atom * (sparse.left.abs().mean() + sparse.right.abs().mean())
        loss = recon + reg_atom

        opt.zero_grad()
        loss.backward()
        opt.step()

        with torch.no_grad():
            sparse.left.data /= sparse.left.data.norm(dim=0, keepdim=True).clamp_min(1e-8)
            sparse.right.data /= sparse.right.data.norm(dim=0, keepdim=True).clamp_min(1e-8)

        if verbose and hasattr(pbar, "set_description"):
            with torch.no_grad():
                sim = sparse.similarity(model).item()
            pbar.set_description(f"recon: {recon.item():.4f} sim: {sim:.4f}")

    torch.set_grad_enabled(False)
    return sparse

sparse_dl = fit_dictionary_learning(model)
metrics_dl = evaluate(sparse_dl, model, test)

with torch.no_grad():
    atom_usage = sparse_dl.down.norm(dim=0)
    unused_frac = (atom_usage < 0.01).float().mean().item()
    atoms_per_class = (sparse_dl.down.abs() > 0.01).float().sum(dim=1).mean().item()

print(metrics_dl)
print(f"unused atoms (||D[:,r]|| < 0.01): {unused_frac:.2%}")
print(f"mean atoms used per class (|D[c,r]| > 0.01): {atoms_per_class:.1f} / {OC_RANK}")

fig_dl = visualize_decomposition(sparse_dl, title="Dictionary learning (overcomplete, unit-norm, Frobenius + L1 on L, R)",
                                  save_path="figures/fig_dictionary_learning.png")
fig_dl.show()

## Scratch — prior sweep beyond Exp 1–5

Exploratory variants not (yet) in `experiment_plans/`: spatial smoothness, distinctness, group lasso, warm-start, and L1-on-D at low rank. Reuses `reconstruct_B_target`, `evaluate`, `visualize_decomposition` from the helpers cell above (which correctly fold `W_E` into `W_L, W_R` — important, this is the bug that was buried in the consolidated v2 cell).

In [ ]:
import os
os.makedirs("figures", exist_ok=True)


def report(sparse_model, name, model, test):
    metrics = evaluate(sparse_model, model, test)
    with torch.no_grad():
        atom_usage = sparse_model.down.norm(dim=0)
        unused_frac = (atom_usage < 0.01).float().mean().item()
        atoms_per_class = (sparse_model.down.abs() > 0.01).float().sum(dim=1).mean().item()
    summary = {
        "name": name,
        "sim": metrics["similarity"],
        "orig_acc": metrics["orig_acc"],
        "sparse_acc": metrics["sparse_acc"],
        "unused_frac": unused_frac,
        "atoms_per_class": atoms_per_class,
    }
    print(f"  {name:<30} sim={summary['sim']:.4f}  acc={summary['sparse_acc']:.4f}  "
          f"unused={unused_frac*100:.1f}%  atoms/cls={atoms_per_class:.1f}")
    return summary


def spatial_smoothness(L):
    L_img = L.view(28, 28, -1)
    dh = (L_img[1:, :, :] - L_img[:-1, :, :]).pow(2).mean()
    dw = (L_img[:, 1:, :] - L_img[:, :-1, :]).pow(2).mean()
    return dh + dw


def distinctness_penalty(L):
    L_normed = L / (L.norm(dim=0, keepdim=True) + 1e-8)
    gram = L_normed.T @ L_normed
    off_diagonal = gram - torch.eye(gram.shape[0], device=gram.device)
    return off_diagonal.pow(2).mean()


def group_lasso(L, R, D):
    norms = (L.pow(2).sum(0) + R.pow(2).sum(0) + D.pow(2).sum(0)).sqrt()
    return norms.sum()


In [ ]:
def fit_cp(
    model,
    oc_rank=128,
    n_steps=400,
    lr=1e-2,
    seed=0,
    alpha_atom=0.0,
    alpha_code=0.0,
    gamma_smooth=0.0,
    delta_distinct=0.0,
    lambda_group=0.0,
    tie_lr=False,
    nonneg=False,
    desc="fit",
    verbose=True,
):
    """Unified fitting function for all CP variants. All priors disabled by default.

    Uses `reconstruct_B_target` from the helpers cell — that one correctly folds
    `W_E` into `W_L, W_R`, giving a (10, 784, 784) pixel-space tensor that matches
    `Sparse.tensor()`. The local copy that lived in the old consolidated cell
    omitted `W_E` and produced a (10, 256, 256) embedding-space tensor — hence
    the RuntimeError at `B_target - B_hat`."""
    torch.manual_seed(seed)
    sparse = Sparse.from_config(rank=oc_rank).to(device)
    B_target = reconstruct_B_target(model).detach()

    if tie_lr:
        params = [sparse.left, sparse.down]
    else:
        params = list(sparse.parameters())
    opt = torch.optim.Adam(params, lr=lr)

    torch.set_grad_enabled(True)
    pbar = tqdm(range(n_steps), desc=desc) if verbose else range(n_steps)

    for _ in pbar:
        if tie_lr:
            with torch.no_grad():
                sparse.right.data = sparse.left.data.clone()

        if nonneg:
            L_eff = sparse.left ** 2
            R_eff = sparse.right ** 2 if not tie_lr else L_eff
            B_hat = torch.einsum("ir,jr,cr->cij", L_eff, R_eff, sparse.down)
            B_hat = 0.5 * (B_hat + B_hat.transpose(-1, -2))
        else:
            B_hat = sparse.tensor()

        recon = (B_target - B_hat).pow(2).mean()
        loss = recon

        L_use = (sparse.left ** 2) if nonneg else sparse.left
        R_use = (sparse.right ** 2) if nonneg else sparse.right
        if alpha_atom > 0:
            loss = loss + alpha_atom * (L_use.abs().mean() + R_use.abs().mean())
        if alpha_code > 0:
            loss = loss + alpha_code * sparse.down.abs().mean()
        if gamma_smooth > 0:
            loss = loss + gamma_smooth * (spatial_smoothness(L_use) + spatial_smoothness(R_use))
        if delta_distinct > 0:
            loss = loss + delta_distinct * distinctness_penalty(L_use)
        if lambda_group > 0:
            loss = loss + lambda_group * group_lasso(L_use, R_use, sparse.down)

        opt.zero_grad()
        loss.backward()
        opt.step()

        if verbose and hasattr(pbar, "set_description"):
            with torch.no_grad():
                sim = sparse.similarity(model).item()
            pbar.set_description(f"{desc} recon: {recon.item():.4f} sim: {sim:.4f}")

    with torch.no_grad():
        if tie_lr:
            sparse.right.data = sparse.left.data.clone()
        if nonneg:
            sparse.left.data = (sparse.left ** 2).data
            sparse.right.data = (sparse.right ** 2).data

    torch.set_grad_enabled(False)
    return sparse


In [ ]:
def fit_warm_start(model, sparse_init, alpha_atom=0.0001, alpha_code=0.0001,
                    n_steps=200, lr=1e-3, desc="warmstart"):
    sparse = Sparse.from_config(rank=sparse_init.left.shape[1]).to(device)
    with torch.no_grad():
        sparse.left.data = sparse_init.left.data.clone()
        sparse.right.data = sparse_init.right.data.clone()
        sparse.down.data = sparse_init.down.data.clone()

    B_target = reconstruct_B_target(model).detach()
    opt = torch.optim.Adam(sparse.parameters(), lr=lr)

    torch.set_grad_enabled(True)
    pbar = tqdm(range(n_steps), desc=desc)
    for _ in pbar:
        B_hat = sparse.tensor()
        recon = (B_target - B_hat).pow(2).mean()
        reg_atom = alpha_atom * (sparse.left.abs().mean() + sparse.right.abs().mean())
        reg_code = alpha_code * sparse.down.abs().mean()
        loss = recon + reg_atom + reg_code
        opt.zero_grad()
        loss.backward()
        opt.step()
        with torch.no_grad():
            sim = sparse.similarity(model).item()
        pbar.set_description(f"{desc} recon: {recon.item():.4f} sim: {sim:.4f}")
    torch.set_grad_enabled(False)
    return sparse


### Scratch-Exp 1 — Dictionary learning baseline (L1 on L, R only)

In [ ]:
all_results = []
all_sparse_models = {}

sparse_dl = fit_cp(model, oc_rank=128, alpha_atom=0.0001, desc="dict_learning")
all_sparse_models["dict_learning"] = sparse_dl
all_results.append(report(sparse_dl, "dict_learning", model, test))
fig = visualize_decomposition(sparse_dl,
    title="Dictionary learning (L1 on L,R, rank=128)",
    save_path="figures/fig_scratch_01_dict_learning.png")
fig.show()


### Scratch-Exp 2 — Prior sweep at rank=128

Six priors at fixed rank: L1 on L/R, L1 on D, both, spatial smoothness, distinctness, group lasso.

In [ ]:
priors = [
    ("L1_LR",        dict(alpha_atom=0.0001)),
    ("L1_D",         dict(alpha_code=0.0001)),
    ("L1_LR_and_D",  dict(alpha_atom=0.0001, alpha_code=0.0001)),
    ("smoothness",   dict(gamma_smooth=0.001)),
    ("distinctness", dict(delta_distinct=0.001)),
    ("group_lasso",  dict(lambda_group=0.0001)),
]

for name, kwargs in priors:
    print(f"\n--- {name} ---")
    sp = fit_cp(model, oc_rank=128, desc=name, **kwargs)
    all_sparse_models[name] = sp
    all_results.append(report(sp, name, model, test))
    fig = visualize_decomposition(sp,
        title=f"Prior: {name}",
        save_path=f"figures/fig_scratch_02_prior_{name}.png")
    fig.show()


### Scratch-Exp 3a — Symmetric CP (`L=R`) + L1, rank=16

In [ ]:
sparse_sym = fit_cp(model, oc_rank=16, alpha_atom=0.0001, alpha_code=0.0001,
                     tie_lr=True, desc="symmetric")
all_sparse_models["symmetric"] = sparse_sym
all_results.append(report(sparse_sym, "symmetric (L=R)", model, test))
fig = visualize_decomposition(sparse_sym,
    title="Symmetric CP (L=R, rank=16)",
    save_path="figures/fig_scratch_03_symmetric.png")
fig.show()


### Scratch-Exp 3b — Non-negative (`L_eff = L²`), rank=8

In [ ]:
sparse_nn = fit_cp(model, oc_rank=8, alpha_atom=0.0001, alpha_code=0.0001,
                    nonneg=True, desc="nonneg")
all_sparse_models["nonneg"] = sparse_nn
all_results.append(report(sparse_nn, "nonneg (L²)", model, test))
fig = visualize_decomposition(sparse_nn,
    title="Non-negative (L²) + rank=8",
    save_path="figures/fig_scratch_04_nonneg.png")
fig.show()


### Scratch-Exp 4 — Warm-start: dict_learning → add L1 on D

In [ ]:
sparse_warm = fit_warm_start(model, sparse_dl, alpha_atom=0.0001, alpha_code=0.0001,
                              n_steps=200, lr=1e-3, desc="warmstart")
all_sparse_models["warmstart"] = sparse_warm
all_results.append(report(sparse_warm, "warmstart dl + L1_D", model, test))
fig = visualize_decomposition(sparse_warm,
    title="Warm-start: dict_learning + L1 on D",
    save_path="figures/fig_scratch_05_warmstart.png")
fig.show()


### Scratch-Exp 5 — L1 on D at rank=16 (forced sharing)

In [ ]:
sparse_best = fit_cp(model, oc_rank=16, alpha_code=0.0001, desc="L1_D_rank16")
all_sparse_models["L1_D_rank16"] = sparse_best
all_results.append(report(sparse_best, "L1_D rank=16 (best)", model, test))
fig = visualize_decomposition(sparse_best,
    title="L1 on D, rank=16 (forced sharing)",
    save_path="figures/fig_scratch_06_L1_D_rank16.png")
fig.show()


### Scratch summary

In [ ]:
print("=" * 90)
print(f"{'variant':<28} {'sim':<8} {'sparse_acc':<12} {'unused%':<10} {'atoms/cls':<10}")
print("-" * 90)
for r in all_results:
    print(f"{r['name']:<28} {r['sim']:<8.4f} {r['sparse_acc']:<12.4f} "
          f"{r['unused_frac']*100:<10.1f} {r['atoms_per_class']:<10.1f}")

print("\nFigures: figures/fig_scratch_*.png  |  Models: all_sparse_models")


In [ ]:
def fit_warm_start(model, sparse_init, alpha_atom=0.0001, alpha_code=0.0001,
                    gamma_smooth=0.0,
                    n_steps=200, lr=1e-3, desc="warmstart"):
    sparse = Sparse.from_config(rank=sparse_init.left.shape[1]).to(device)
    with torch.no_grad():
        sparse.left.data = sparse_init.left.data.clone()
        sparse.right.data = sparse_init.right.data.clone()
        sparse.down.data = sparse_init.down.data.clone()

    B_target = reconstruct_B_target(model).detach()
    opt = torch.optim.Adam(sparse.parameters(), lr=lr)

    torch.set_grad_enabled(True)
    pbar = tqdm(range(n_steps), desc=desc)
    for _ in pbar:
        B_hat = sparse.tensor()
        recon = (B_target - B_hat).pow(2).mean()
        reg_atom = alpha_atom * (sparse.left.abs().mean() + sparse.right.abs().mean())
        reg_code = alpha_code * sparse.down.abs().mean()
        reg_smooth = gamma_smooth * (spatial_smoothness(sparse.left)
                                   + spatial_smoothness(sparse.right))
        loss = recon + reg_atom + reg_code + reg_smooth
        opt.zero_grad()
        loss.backward()
        opt.step()
        with torch.no_grad():
            sim = sparse.similarity(model).item()
        pbar.set_description(f"{desc} recon: {recon.item():.4f} sim: {sim:.4f}")
    torch.set_grad_enabled(False)
    return sparse


sparse_warm_smooth = fit_warm_start(model, sparse_dl,
                                     alpha_atom=0.0001, alpha_code=0.01,
                                     gamma_smooth=1e-4,
                                     n_steps=200, lr=1e-3, desc="warmstart+smooth")
all_sparse_models["warmstart_smooth"] = sparse_warm_smooth
all_results.append(report(sparse_warm_smooth, "warmstart dl + L1_D + smooth", model, test))
fig = visualize_decomposition(sparse_warm_smooth,
    title="Warm-start: dict_learning + L1 on D + spatial smoothness",
    save_path="figures/fig_scratch_07_warmstart_smooth.png")
fig.show()


In [ ]:
sparse_warm_smooth = fit_warm_start(model, sparse_dl,
                                     alpha_atom=0.0, alpha_code=0.0,
                                     gamma_smooth=1e-4,
                                     n_steps=200, lr=1e-3, desc="warmstart+smooth")
all_sparse_models["warmstart_smooth"] = sparse_warm_smooth
all_results.append(report(sparse_warm_smooth, "warmstart dl + smooth only", model, test))
fig = visualize_decomposition(sparse_warm_smooth,
    title="Warm-start: dict_learning + spatial smoothness only",
    save_path="figures/fig_scratch_07_warmstart_smooth.png")
fig.show()


In [ ]:
sparse_parts = fit_cp(
    model,
    oc_rank=16,          # low rank forces sharing
    nonneg=True,         # NMF-style parts-based
    alpha_atom=1e-3,     # L1 on L,R for localization
    alpha_code=1e-3,     # L1 on D for class specialization
    gamma_smooth=1e-3,   # connected shapes
    n_steps=1000,
    lr=5e-3,
    seed=0,
    desc="parts-based",
)
all_sparse_models["parts_based"] = sparse_parts
all_results.append(report(sparse_parts, "parts_based", model, test))
fig = visualize_decomposition(sparse_parts, title="Low-rank + non-neg + L1 + smooth",
                              save_path="figures/fig_scratch_08_parts_based.png")
fig.show()


In [ ]:
sparse_parts = fit_cp(
    model, oc_rank=16, nonneg=True,
    alpha_atom=1e-3, alpha_code=1e-3, gamma_smooth=1e-3,
    n_steps=1000,
    lr=1e-3,
    seed=0, desc="parts-based",
)
all_sparse_models["parts_based"] = sparse_parts
all_results.append(report(sparse_parts, "parts_based", model, test))
fig = visualize_decomposition(sparse_parts, title="Low-rank + non-neg + L1 + smooth (lr=1e-3)",
                              save_path="figures/fig_scratch_08_parts_based.png")
fig.show()
